<a href="https://colab.research.google.com/github/sowjanyageethika-lgtm/material-stress-analyzer/blob/main/material_stress_analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# ============================================================
# MATERIAL STRESS ANALYZER
# Google Colab Project
# ============================================================

!pip -q install plotly ipywidgets

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display, HTML, Markdown
import ipywidgets as widgets
# ============================================================
# MATERIAL STRESS ANALYZER - CORE ENGINE
# ============================================================

# -----------------------------
# USER INPUT
# -----------------------------

# Stress tensor in MPa
sigma = np.array([
    [120, 40, 20],
    [40,  80, 30],
    [20,  30, 50]
], dtype=float)

# Material properties
yield_strength = 250.0   # MPa

# -----------------------------
# CHECK SYMMETRY
# -----------------------------

if not np.allclose(sigma, sigma.T):
    raise ValueError("Stress matrix must be symmetric.")

# -----------------------------
# EIGENVALUE / EIGENVECTOR
# -----------------------------

eigenvalues, eigenvectors = np.linalg.eigh(sigma)

# Sort from maximum to minimum
idx = np.argsort(eigenvalues)[::-1]

principal_stresses = eigenvalues[idx]
principal_directions = eigenvectors[:, idx]

sigma1, sigma2, sigma3 = principal_stresses

# -----------------------------
# STRESS CALCULATIONS
# -----------------------------

# Maximum shear stress
tau_max = (sigma1 - sigma3) / 2

# Von Mises stress
von_mises = np.sqrt(
    ((sigma1 - sigma2)**2 +
     (sigma2 - sigma3)**2 +
     (sigma3 - sigma1)**2) / 2
)

# Factor of safety
factor_of_safety = yield_strength / von_mises

# Hydrostatic / mean stress
mean_stress = np.mean(principal_stresses)

# Stress tensor invariants
I1 = np.trace(sigma)

I2 = (
    sigma[0,0]*sigma[1,1] +
    sigma[1,1]*sigma[2,2] +
    sigma[2,2]*sigma[0,0]
    - sigma[0,1]**2
    - sigma[1,2]**2
    - sigma[0,2]**2
)

I3 = np.linalg.det(sigma)

# -----------------------------
# SAFETY CLASSIFICATION
# -----------------------------

if factor_of_safety >= 2:
    status = "SAFE"
    status_color = "#00C853"
elif factor_of_safety >= 1:
    status = "CAUTION"
    status_color = "#FFB300"
else:
    status = "FAILURE RISK"
    status_color = "#FF1744"

# -----------------------------
# DISPLAY RESULTS
# -----------------------------

print("=" * 65)
print("              MATERIAL STRESS ANALYZER")
print("=" * 65)

print("\nINPUT STRESS MATRIX (MPa)")
print(pd.DataFrame(
    sigma,
    index=["X", "Y", "Z"],
    columns=["X", "Y", "Z"]
).round(3))

print("\nPRINCIPAL STRESSES")
print(f"σ1 = {sigma1:.3f} MPa")
print(f"σ2 = {sigma2:.3f} MPa")
print(f"σ3 = {sigma3:.3f} MPa")

print("\nPRINCIPAL DIRECTIONS")
for i in range(3):
    print(
        f"Direction {i+1}: "
        f"[{principal_directions[0,i]:.4f}, "
        f"{principal_directions[1,i]:.4f}, "
        f"{principal_directions[2,i]:.4f}]"
    )

print("\nENGINEERING PARAMETERS")
print(f"Maximum Shear Stress = {tau_max:.3f} MPa")
print(f"Von Mises Stress     = {von_mises:.3f} MPa")
print(f"Mean Stress          = {mean_stress:.3f} MPa")
print(f"Yield Strength       = {yield_strength:.3f} MPa")
print(f"Factor of Safety     = {factor_of_safety:.3f}")

print("\nSTRESS INVARIANTS")
print(f"I1 = {I1:.3f}")
print(f"I2 = {I2:.3f}")
print(f"I3 = {I3:.3f}")

print("\nSTATUS:", status)
print("=" * 65)

# ============================================================
# INDUSTRY STYLE DASHBOARD
# ============================================================

dashboard = f"""
<div style="
    background: linear-gradient(135deg,#07111f,#102a43);
    color:white;
    padding:30px;
    border-radius:20px;
    font-family:Arial;
    box-shadow:0px 10px 30px rgba(0,0,0,0.3);
">

<h1 style="text-align:center;color:#00e5ff;">
    MATERIAL STRESS ANALYZER
</h1>

<p style="text-align:center;color:#b8c7d9;">
    Principal Stress & Direction Analysis | Engineering Simulation
</p>

<hr style="border-color:#31506b;">

<div style="display:flex;gap:15px;justify-content:center;flex-wrap:wrap;">

<div style="background:#102d44;padding:20px;border-radius:15px;width:180px;">
<h4>Principal Stress σ₁</h4>
<h2 style="color:#00e5ff;">{sigma1:.2f} MPa</h2>
</div>

<div style="background:#102d44;padding:20px;border-radius:15px;width:180px;">
<h4>Principal Stress σ₂</h4>
<h2 style="color:#7cfcff;">{sigma2:.2f} MPa</h2>
</div>

<div style="background:#102d44;padding:20px;border-radius:15px;width:180px;">
<h4>Principal Stress σ₃</h4>
<h2 style="color:#7cfcff;">{sigma3:.2f} MPa</h2>
</div>

<div style="background:#102d44;padding:20px;border-radius:15px;width:180px;">
<h4>Von Mises</h4>
<h2 style="color:#ffcc00;">{von_mises:.2f} MPa</h2>
</div>

<div style="background:#102d44;padding:20px;border-radius:15px;width:180px;">
<h4>Safety Factor</h4>
<h2 style="color:{status_color};">{factor_of_safety:.2f}</h2>
</div>

</div>

<br>

<div style="
    background:{status_color};
    padding:15px;
    border-radius:12px;
    text-align:center;
    font-size:22px;
    font-weight:bold;
">
STATUS: {status}
</div>

</div>
"""

display(HTML(dashboard))

# ============================================================
# 3D PRINCIPAL STRESS DIRECTIONS
# ============================================================

fig = go.Figure()

colors = ["#ff1744", "#00e5ff", "#00ff88"]
names = ["σ1 - Maximum", "σ2 - Intermediate", "σ3 - Minimum"]

# Origin
origin = np.array([0, 0, 0])

for i in range(3):

    direction = principal_directions[:, i]

    # Scale arrows according to stress magnitude
    scale = abs(principal_stresses[i]) / abs(sigma1)

    end = direction * scale

    # Arrow line
    fig.add_trace(go.Scatter3d(
        x=[0, end[0]],
        y=[0, end[1]],
        z=[0, end[2]],
        mode="lines+markers",
        line=dict(
            color=colors[i],
            width=10
        ),
        marker=dict(
            size=6,
            color=colors[i]
        ),
        name=names[i]
    ))

    # Direction label
    fig.add_trace(go.Scatter3d(
        x=[end[0]*1.15],
        y=[end[1]*1.15],
        z=[end[2]*1.15],
        mode="text",
        text=[
            f"σ{i+1} = {principal_stresses[i]:.1f} MPa"
        ],
        textfont=dict(
            size=16,
            color=colors[i]
        ),
        showlegend=False
    ))

# Coordinate axes

axis_length = 1.25

axes = [
    ([0, axis_length], [0,0], [0,0], "X", "white"),
    ([0,0], [0, axis_length], [0,0], "Y", "yellow"),
    ([0,0], [0,0], [0, axis_length], "Z", "lime")
]

for x,y,z,label,color in axes:

    fig.add_trace(go.Scatter3d(
        x=x,
        y=y,
        z=z,
        mode="lines+text",
        line=dict(color=color, width=5),
        text=["", label],
        textfont=dict(size=18),
        showlegend=False
    ))

fig.update_layout(
    title="3D Principal Stress Directions",
    template="plotly_dark",
    width=900,
    height=700,
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
        aspectmode="cube"
    ),
    legend=dict(
        x=0.02,
        y=0.98
    )
)

fig.show()
# ============================================================
# 3D STRESS FIELD
# ============================================================

theta = np.linspace(0, np.pi, 50)
phi = np.linspace(0, 2*np.pi, 100)

theta_grid, phi_grid = np.meshgrid(theta, phi)

x = np.sin(theta_grid) * np.cos(phi_grid)
y = np.sin(theta_grid) * np.sin(phi_grid)
z = np.cos(theta_grid)

normal_stress = np.zeros_like(x)

for i in range(x.shape[0]):
    for j in range(x.shape[1]):

        n = np.array([
            x[i,j],
            y[i,j],
            z[i,j]
        ])

        normal_stress[i,j] = n @ sigma @ n

fig = go.Figure()

fig.add_trace(go.Surface(
    x=x,
    y=y,
    z=z,
    surfacecolor=normal_stress,
    colorscale="Turbo",
    colorbar=dict(
        title="Normal Stress<br>(MPa)"
    ),
    hovertemplate=
        "X: %{x:.2f}<br>" +
        "Y: %{y:.2f}<br>" +
        "Z: %{z:.2f}<br>" +
        "Stress: %{surfacecolor:.2f} MPa"
))

fig.update_layout(
    title="3D Directional Stress Field",
    template="plotly_dark",
    width=900,
    height=700,
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
        aspectmode="cube"
    )
)

fig.show()
# ============================================================
# ANIMATED STRESS PROBE
# ============================================================

frames = []

N = 60

for k in range(N):

    angle = 2*np.pi*k/N

    # Moving direction
    n = np.array([
        np.cos(angle),
        np.sin(angle),
        0.5*np.sin(2*angle)
    ])

    n = n / np.linalg.norm(n)

    # Normal stress
    normal = n @ sigma @ n

    # Traction vector
    traction = sigma @ n

    frame = go.Frame(
        data=[
            go.Scatter3d(
                x=[0, n[0]],
                y=[0, n[1]],
                z=[0, n[2]],
                mode="lines+markers",
                line=dict(
                    color="#00ffea",
                    width=12
                ),
                marker=dict(
                    size=8,
                    color="#ffea00"
                )
            ),

            go.Scatter3d(
                x=[n[0], n[0] + traction[0]/150],
                y=[n[1], n[1] + traction[1]/150],
                z=[n[2], n[2] + traction[2]/150],
                mode="lines",
                line=dict(
                    color="#ff1744",
                    width=10
                )
            )
        ],
        name=str(k)
    )

    frames.append(frame)

fig = go.Figure(
    data=[
        go.Scatter3d(
            x=[0,1],
            y=[0,0],
            z=[0,0],
            mode="lines+markers",
            line=dict(
                color="#00ffea",
                width=12
            ),
            marker=dict(
                size=8,
                color="#ffea00"
            )
        ),

        go.Scatter3d(
            x=[1,1.5],
            y=[0,0],
            z=[0,0],
            mode="lines",
            line=dict(
                color="#ff1744",
                width=10
            )
        )
    ],
    frames=frames
)

fig.update_layout(
    title="Animated Stress Direction Probe",
    template="plotly_dark",
    width=900,
    height=700,

    scene=dict(
        xaxis=dict(range=[-1.5,1.5]),
        yaxis=dict(range=[-1.5,1.5]),
        zaxis=dict(range=[-1.5,1.5]),
        aspectmode="cube"
    ),

    updatemenus=[
        dict(
            type="buttons",
            showactive=False,
            buttons=[
                dict(
                    label="▶ START",
                    method="animate",
                    args=[
                        None,
                        dict(
                            frame=dict(
                                duration=80,
                                redraw=True
                            ),
                            fromcurrent=True,
                            transition=dict(duration=0)
                        )
                    ]
                ),

                dict(
                    label="⏸ PAUSE",
                    method="animate",
                    args=[
                        [None],
                        dict(
                            frame=dict(duration=0),
                            mode="immediate"
                        )
                    ]
                )
            ]
        )
    ]
)

fig.show()
# ============================================================
# 3D MOHR'S CIRCLES
# ============================================================

pairs = [
    (sigma1, sigma2, "σ1 - σ2"),
    (sigma2, sigma3, "σ2 - σ3"),
    (sigma1, sigma3, "σ1 - σ3")
]

fig = go.Figure()

colors = ["#ff1744", "#00e5ff", "#00ff88"]

for (s_a, s_b, name), color in zip(pairs, colors):

    center = (s_a + s_b) / 2
    radius = abs(s_a - s_b) / 2

    theta = np.linspace(0, np.pi, 200)

    x = center + radius*np.cos(theta)
    y = radius*np.sin(theta)

    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode="lines",
        name=name,
        line=dict(
            color=color,
            width=4
        )
    ))

# Principal stress points
fig.add_trace(go.Scatter(
    x=[sigma1, sigma2, sigma3],
    y=[0,0,0],
    mode="markers+text",
    text=["σ1","σ2","σ3"],
    textposition="bottom center",
    marker=dict(
        size=12,
        color="white"
    ),
    name="Principal Stresses"
))

fig.update_layout(
    title="3D Mohr's Stress Circles",
    template="plotly_dark",
    xaxis_title="Normal Stress σ (MPa)",
    yaxis_title="Shear Stress τ (MPa)",
    width=900,
    height=650,
    xaxis=dict(
        zeroline=True,
        showgrid=True
    ),
    yaxis=dict(
        zeroline=True,
        showgrid=True
    )
)

fig.show()
# ============================================================
# ENGINEERING REPORT
# ============================================================

report = pd.DataFrame({
    "Parameter": [
        "σ1 - Maximum Principal Stress",
        "σ2 - Intermediate Principal Stress",
        "σ3 - Minimum Principal Stress",
        "Maximum Shear Stress",
        "Von Mises Stress",
        "Mean Stress",
        "Material Yield Strength",
        "Factor of Safety",
        "Status"
    ],

    "Value": [
        sigma1,
        sigma2,
        sigma3,
        tau_max,
        von_mises,
        mean_stress,
        yield_strength,
        factor_of_safety,
        status
    ],

    "Unit": [
        "MPa",
        "MPa",
        "MPa",
        "MPa",
        "MPa",
        "MPa",
        "MPa",
        "-",
        "-"
    ]
})

display(report.style
        .format({
            "Value": lambda x: f"{x:.3f}" if isinstance(x,(float,np.floating)) else x
        })
        .set_caption("MATERIAL STRESS ANALYZER - ENGINEERING REPORT"))

# Save report
report.to_csv("material_stress_report.csv", index=False)

print("\nReport saved as: material_stress_report.csv")
# ============================================================
# INTERACTIVE STRESS MATRIX
# ============================================================

def analyze_interactive(
    sxx=120,
    syy=80,
    szz=50,
    txy=40,
    tyz=30,
    txz=20
):

    S = np.array([
        [sxx, txy, txz],
        [txy, syy, tyz],
        [txz, tyz, szz]
    ], dtype=float)

    vals, vecs = np.linalg.eigh(S)

    idx = np.argsort(vals)[::-1]

    vals = vals[idx]
    vecs = vecs[:,idx]

    s1, s2, s3 = vals

    vm = np.sqrt(
        ((s1-s2)**2 +
         (s2-s3)**2 +
         (s3-s1)**2)/2
    )

    tau = (s1-s3)/2

    fos = yield_strength/vm

    if fos >= 2:
        state = "SAFE"
        color = "green"
    elif fos >= 1:
        state = "CAUTION"
        color = "orange"
    else:
        state = "FAILURE RISK"
        color = "red"

    display(HTML(f"""
    <div style="
        background:#07111f;
        color:white;
        padding:25px;
        border-radius:15px;
        text-align:center;
    ">

    <h2 style="color:#00e5ff;">
    LIVE STRESS ANALYSIS
    </h2>

    <h3>
    σ₁ = {s1:.2f} MPa &nbsp;&nbsp;
    σ₂ = {s2:.2f} MPa &nbsp;&nbsp;
    σ₃ = {s3:.2f} MPa
    </h3>

    <p>Von Mises Stress:
    <b>{vm:.2f} MPa</b></p>

    <p>Maximum Shear:
    <b>{tau:.2f} MPa</b></p>

    <p>Factor of Safety:
    <b>{fos:.2f}</b></p>

    <h2 style="color:{color};">
    {state}
    </h2>

    </div>
    """))


style = {'description_width': 'initial'}

widgets.interact(
    analyze_interactive,

    sxx=widgets.FloatSlider(
        value=120,
        min=-300,
        max=300,
        step=5,
        description="σxx (MPa)",
        style=style
    ),

    syy=widgets.FloatSlider(
        value=80,
        min=-300,
        max=300,
        step=5,
        description="σyy (MPa)",
        style=style
    ),

    szz=widgets.FloatSlider(
        value=50,
        min=-300,
        max=300,
        step=5,
        description="σzz (MPa)",
        style=style
    ),

    txy=widgets.FloatSlider(
        value=40,
        min=-200,
        max=200,
        step=5,
        description="τxy (MPa)",
        style=style
    ),

    tyz=widgets.FloatSlider(
        value=30,
        min=-200,
        max=200,
        step=5,
        description="τyz (MPa)",
        style=style
    ),

    txz=widgets.FloatSlider(
        value=20,
        min=-200,
        max=200,
        step=5,
        description="τxz (MPa)",
        style=style
    )
);





              MATERIAL STRESS ANALYZER

INPUT STRESS MATRIX (MPa)
       X     Y     Z
X  120.0  40.0  20.0
Y   40.0  80.0  30.0
Z   20.0  30.0  50.0

PRINCIPAL STRESSES
σ1 = 155.166 MPa
σ2 = 63.647 MPa
σ3 = 31.187 MPa

PRINCIPAL DIRECTIONS
Direction 1: [-0.7857, -0.5392, -0.3032]
Direction 2: [-0.6149, 0.6273, 0.4779]
Direction 3: [0.0674, -0.5619, 0.8244]

ENGINEERING PARAMETERS
Maximum Shear Stress = 61.989 MPa
Von Mises Stress     = 111.355 MPa
Mean Stress          = 83.333 MPa
Yield Strength       = 250.000 MPa
Factor of Safety     = 2.245

STRESS INVARIANTS
I1 = 250.000
I2 = 16700.000
I3 = 308000.000

STATUS: SAFE


,Parameter,Value,Unit
0,σ1 - Maximum Principal Stress,155.166,MPa
1,σ2 - Intermediate Principal Stress,63.647,MPa
2,σ3 - Minimum Principal Stress,31.187,MPa
3,Maximum Shear Stress,61.989,MPa
4,Von Mises Stress,111.355,MPa
5,Mean Stress,83.333,MPa
6,Material Yield Strength,250.000,MPa
7,Factor of Safety,2.245,-
8,Status,SAFE,-



Report saved as: material_stress_report.csv


interactive(children=(FloatSlider(value=120.0, description='σxx (MPa)', max=300.0, min=-300.0, step=5.0, style…